[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/riccati/riccati.ipynb)

# Docking by Riccati feedback

A small vessel has three reversible thrusters and a deadline to reach a target pose. Pushing harder gets it there sooner but costs more effort. Working backward from the final landing, the Riccati recursion finds a feedback map from the remaining displacement and turn to the force and torque to apply. Two effort penalties give two approaches, computed together.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np
from IPython.display import Image, display

from numga import NumpyContext, concatenate, stack
from numga.algebras import PGA2D
from examples.animation import save_animation
from examples.mechanics.riccati import render

np.set_printoptions(precision=4, suppress=True)

ga = PGA2D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Point = ga.gatype.antivector()
Motor = ga.gatype.rotor()
Twist = ga.gatype.bivector()
Forque = ga.gatype.antibivector()
# Every cost is a quadratic form, a scalar with two open slots: `state_cost(error, error)`.
StateCost = ga.gatype((Scalar, Twist, Twist))           # Scalar <- (Twist, Twist)
EffortCost = ga.gatype((Scalar, Forque, Forque))        # Scalar <- (Forque, Forque)
Dynamics = ga.gatype((Twist, Twist))                   # Twist <- Twist
Actuation = ga.gatype((Twist, Forque))                  # Twist <- Forque
Feedback = ga.gatype((Forque, Twist))                   # Forque <- Twist

labels = ("Lower effort penalty", "Higher effort penalty")
cases = len(labels)
effort_scales = np.array([0.05, 5.])                                            # [cases]
thruster_weights = np.array([1., 2., 3.])                                       # [thrusters]
# Drag along the forward and sideways directions through the centre, and against turning.
drag_weights = np.array([2., 3., 1.])                                           # [modes]
# Rows: bow and stern; columns: forward and sideways displacement. Favor the bow and sideways alignment.
tracking_weights = np.array([[4., 8.], [1., 2.]])                               # [sites, directions]
dt = 0.1
steps = 72
# How much stiffer the pose is priced at the deadline than along the way: the vessel must be there.
landing = 1e6
frame_duration = 40
arrow_scale = 0.04

## 1. Lines of force and a pose error

Two thrusters push forward from opposite sides of the stern; the third pushes sideways at the bow. Joining each mounting point to its direction gives a **forque**: a line carrying both the force and its turning effect. A **twist** carries translation and turn. Their regressive product, `forque & twist`, measures work. The two purple points mark the hull locations used to measure docking error.

In [ ]:
hull = mv.antivector([[-.45, -.23, 1], [.28, -.23, 1], [.55, 0, 1],
                      [.28, .23, 1], [-.45, .23, 1]])                           # [vertices] Point
mounts = mv.antivector([[-.3, .2, 1], [-.3, -.2, 1], [.35, 0, 1]])              # [thrusters] Point
directions = mv.antivector([[1, 0, 0], [1, 0, 0], [0, 1, 0]])                   # [thrusters] Point at infinity
# Penalize displacement of these two hull points from their locations in the target pose.
# One sits near the bow, the other near the stern: their separation makes heading matter too.
tracking_points = mv.antivector([[.45, 0, 1], [-.35, 0, 1]])                    # [sites] Point
thrusters = mounts & directions                                                 # [thrusters] Forque

# The target is the identity pose; exp(-error / 2) places the vessel relative to it.
# xw and yw slide the hull; xy turns it. Both cases start with the same error.
initial = (mv.xw * .4 + mv.yw * .5 + mv.xy * .35) * np.ones(cases)              # [cases] Twist
render.draw_setup(hull, mounts, directions, tracking_points);

## 2. What does a push cost?

Every cost is a quadratic form: a scalar with two open slots, filled by the same twist or push. Each thruster pays its weight times its squared command. Their weighted dyads add to an authority map; its inverse, `resistance`, gives the minimum effort needed for a requested net push, and `push & resistance(push)` with both pushes open is the effort form. At each tracking point, two lines read displacement along two directions; each reading squared is that displacement's cost, and their weighted sum is the pose cost. Strong drag lets us neglect inertia: mobility turns force and torque directly into velocity. The step model is local about the target, with unrestricted signed thruster commands.

In [ ]:
# An open Twist makes each work pairing a linear readout; the dyad returns it along its force line.
# Dividing by the command cost gives cheaper thrusters more authority in their force direction.
authority = (thrusters * (thrusters & Twist) / thruster_weights).sum(axis=0)    # [] Forque <- Twist
# The inverse prices a requested net force and torque using the available thrusters together.
# This prices thrust effort; the drag map below describes how the vessel moves.
resistance = authority.inverse()                                                # [] Twist <- Forque
# The least effort a push needs, with both pushes open.
effort_cost = (Forque & resistance(Forque)) * effort_scales * dt               # [cases] Scalar <- (Forque, Forque)

# At each hull point, one line reads the forward displacement and one the sideways displacement.
axes = mv.antivector([[1, 0, 0], [0, 1, 0]])                                    # [directions] Point at infinity: forward, sideways
readouts = (tracking_points[:, None] & axes[None, :]) & Twist                   # [sites, directions] Scalar <- Twist
# Each reading squared is that displacement's cost; sum both points and both directions, weighted.
state_cost = (readouts * readouts * tracking_weights).sum(axis=(0, 1)) * dt    # [] Scalar <- (Twist, Twist)

# Drag resists moving along either axis through the centre, and turning: two forces and a pure couple.
centre = mv.antivector([0, 0, 1])                                               # [] Point
drag_lines = concatenate([centre & axes, (axes[0] & axes[1])[None]])            # [modes] Forque
drag = (drag_lines * (drag_lines & Twist) * drag_weights).sum(axis=0)          # [] Forque <- Twist
# Under strong drag a push sets velocity directly; multiplying by dt gives the next pose increment.
mobility = drag.inverse()                                                       # [] Twist <- Forque
# Without a push the vessel stays put.
dynamics = mv.rotor() >> Twist                                                  # [] Twist <- Twist
actuation = mobility * dt                                                       # [] Twist <- Forque

## 3. Work backward to choose the feedback

`value` is a quadratic form on errors: `value(error, error)` is the least cost still to pay. Filling its slots with maps pulls it back through them, so `value(dynamics, actuation)` is the future cost seen from the present error and push, with no transposes. At the deadline, whatever error is left is priced `landing` times as stiffly as along the way, which makes the vessel be there; from it, each step back minimizes over the push. The recursion composes these maps without needing to know how drag or thrusters produced them. The contours show equal remaining cost with heading fixed at the target.

In matrix notation, the update in [the finite-horizon Riccati recursion](https://ee363.stanford.edu/archive/lectures/dlqr.pdf#page=23) reads $H=R+B^\top P B$, $K=-H^{-1}B^\top P A$, and $P_{\mathrm{prev}}=Q+A^\top P(A+BK)$.

In [ ]:
def riccati(value: StateCost, dynamics: Dynamics, actuation: Actuation, state_cost: StateCost,
            effort_cost: EffortCost, steps: int) -> Iterator[tuple[StateCost, Feedback]]:
    """The cost still to pay and the feedback, one more step back from the given cost each time."""
    for _ in range(steps):
        # A push costs effort now and moves the error whose cost is paid next.
        control_cost = effort_cost + value(actuation, actuation)               # [cases] Scalar <- (Forque, Forque)
        # The push that cancels the cost's derivative, for every error at once.
        feedback = -control_cost.solve(value(dynamics, actuation))             # [cases] Forque <- Twist
        # The error's cost now, and its future cost through the step that feedback takes.
        value = state_cost + value(dynamics, dynamics + actuation(feedback))    # [cases] Scalar <- (Twist, Twist)
        yield value, feedback


# From the deadline back, one earlier decision at each iteration.
costs, gains = zip(*riccati(state_cost * landing * np.ones(cases), dynamics, actuation, state_cost,
                            effort_cost, steps))
values = stack(costs)                                                           # [steps, cases] Scalar <- (Twist, Twist): horizons 1 through steps
# The recursion visits later actions first; reverse them into time order.
feedbacks = stack(gains)[::-1]                                                  # [steps, cases] Forque <- Twist

render.draw_costs(values, labels);

## 4. Run forward

Apply each feedback map to the error that actually occurs, then let the push change that error. Both approaches reach the target at the deadline. The lower effort penalty favors getting close sooner; the higher one spreads the effort over more of the available time. The poses below exponentiate the local error twists. Dashed outlines mark the target, and solid outlines sample the approach.

In [ ]:
def rollout(initial: Twist, dynamics: Dynamics, actuation: Actuation,
            feedbacks: Feedback) -> Iterator[Twist]:
    error = initial                                                             # [cases] Twist
    yield error
    for feedback in feedbacks:
        # Apply this time step's policy to the error reached by all preceding pushes.
        push = feedback(error)                                                  # [cases] Forque
        error = dynamics(error) + actuation(push)                              # [cases] Twist
        yield error


errors = stack(list(rollout(initial, dynamics, actuation, feedbacks)))         # [steps + 1, cases] Twist
pushes = feedbacks(errors[:-1])                                                 # [steps, cases] Forque
# Pairing each force line with resistance(pushes) gives its command times its weight.
# After division, the commanded thruster forces add back to the requested net push.
commands = (thrusters & resistance(pushes)[..., None]) / thruster_weights       # [steps, cases, thrusters] Scalar
poses = (errors * -0.5).exp()                                                   # [steps + 1, cases] Motor
render.draw_approaches(hull, poses, labels);

In [ ]:
# Account for the cost paid along the entire approach, and for what is left at the deadline.
tracking_paid = state_cost(errors[:-1], errors[:-1]).sum(axis=0)               # [cases] Scalar
# This cost includes the case's effort penalty; its weight differs between the two approaches.
effort_paid = effort_cost(pushes, pushes).sum(axis=0)                           # [cases] Scalar
landing_paid = state_cost(errors[-1], errors[-1]) * landing                     # [cases] Scalar
# The last backward value prices the full horizon from the common initial error.
predicted = values[-1](initial, initial)                                        # [cases] Scalar
render.print_costs(tracking_paid, effort_paid, landing_paid, predicted, labels)

Orange arrows show the signed thruster commands carried with the hull. Their lengths share one scale in both cases. The controller uses the linear step about the target; the motor display does not add nonlinear hull dynamics to that model.

In [ ]:
# After the final commanded step, show the landed pose with its thrusters off.
display_commands = concatenate((commands, commands[-1:] * 0))                   # [steps + 1, cases, thrusters] Scalar
path = save_animation(render.animate(hull, mounts, directions, poses, display_commands, labels, arrow_scale),
                      "riccati", frame_duration)
display(Image(filename=str(path)))